In [1]:
from pyspark.sql import SparkSession

# Initialize Spark for Machine Learning
spark = SparkSession.builder \
    .appName("FlightDelayPrediction") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Load the Gold dataset from HDFS
gold_data_path = "hdfs://namenode:9000/flight_project/gold/clean_flight_data.parquet"
df = spark.read.parquet(gold_data_path)

print(f"Successfully loaded {df.count()} rows for training!")
df.printSchema()

Successfully loaded 4621247 rows for training!
root
 |-- fl_date: date (nullable = true)
 |-- crs_dep_time: integer (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- crs_arr_time: integer (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- crs_elapsed_time: double (nullable = true)
 |-- distance: double (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- weather_code: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- pressure_msl: double (nullable = true)
 |-- op_unique_carrier_index: double (nullable = true)
 |-- origin_index: double (nullable = true)
 |-- dest_index: double (nullable = true)



In [2]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# 1. Define the features (Ignoring 'fl_date' and the target 'arr_delay')
feature_cols = [
    "crs_dep_time", "dep_delay", "crs_arr_time", "crs_elapsed_time", "distance", 
    "is_weekend", "temperature_2m", "precipitation", "snowfall", "weather_code", 
    "wind_speed_10m", "wind_gusts_10m", "surface_pressure", "pressure_msl", 
    "op_unique_carrier_index", "origin_index", "dest_index"
]

# 2. Assemble features into a single vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

# 3. Define the Model with upgraded maxBins to handle airports/carriers
gbt = GBTRegressor(
    featuresCol="features", 
    labelCol="arr_delay", 
    maxDepth=5, 
    maxIter=50, 
    maxBins=500
)

# 4. Build the Pipeline
pipeline = Pipeline(stages=[assembler, gbt])

# 5. Split the data (80% for training, 20% for testing)
print("Splitting data...")
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# 6. Train the Model! 
print("Training the Gradient Boosting model (this will take a few minutes)...")
model = pipeline.fit(train_data)

# 7. Make predictions and evaluate
print("Making predictions on the test set...")
predictions = model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

print(f"✅ Training Complete! RMSE: {rmse:.2f} minutes")

# Show a sneak peek of the actual vs predicted delays!
predictions.select("arr_delay", "prediction", "dep_delay").show(10)

Splitting data...
Training the Gradient Boosting model (this will take a few minutes)...
Making predictions on the test set...
✅ Training Complete! RMSE: 18.71 minutes
+---------+-------------------+---------+
|arr_delay|         prediction|dep_delay|
+---------+-------------------+---------+
|    -12.0| -13.61664869295476|     -7.0|
|      9.0| 2.2920107665107596|     10.0|
|    -44.0| -21.46009327834496|    -11.0|
|    -24.0|-21.943925591911903|    -11.0|
|      9.0| 14.719689396509313|     21.0|
|    -21.0|-13.318217021949986|    -10.0|
|      1.0|-16.122938241978847|     -5.0|
|     31.0| 29.132397120701302|     32.0|
|     -2.0| -13.08578345742584|     -1.0|
|    -32.0| -17.15656597833446|     -7.0|
+---------+-------------------+---------+
only showing top 10 rows



In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

# Evaluate RMSE
evaluator_rmse = RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# Evaluate MAE
evaluator_mae = RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="mae")
mae = evaluator_mae.evaluate(predictions)

# Evaluate R² (Coefficient of Determination)
evaluator_r2 = RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print("--- Model Performance Report ---")
print(f"🔹 Root Mean Squared Error (RMSE): {rmse:.2f} minutes")
print(f"🔹 Mean Absolute Error (MAE):       {mae:.2f} minutes")
print(f"🔹 R-Squared (R²):                 {r2:.4f}")

--- Model Performance Report ---
🔹 Root Mean Squared Error (RMSE): 18.98 minutes
🔹 Mean Absolute Error (MAE):       9.93 minutes
🔹 R-Squared (R²):                 0.9087


In [3]:
%pip install xgboost pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 1.4 MB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 2.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 MB 1.9 MB/s eta 0:00:0000:0100:05
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession
from xgboost.spark import SparkXGBRegressor

# 1. Initialize Spark Session
spark = (
    SparkSession.builder.appName("FlightDelayPrediction")
    .master("local[*]")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)

df_clean = df
features = [
    "crs_dep_time",
    "crs_arr_time",
    "crs_elapsed_time",
    "distance",
    "dep_delay",
    "is_weekend",
    "temperature_2m",
    "precipitation",
    "snowfall",
    "weather_code",
    "wind_speed_10m",
    "wind_gusts_10m",
    "surface_pressure",
    "pressure_msl",
    "op_unique_carrier_index",
    "origin_index",
    "dest_index",
]


# 4. Train / Test Split
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

# 5. Build Pipeline Stages
assembler = VectorAssembler(
    inputCols=features, outputCol="features", handleInvalid="skip"
)

xgb = SparkXGBRegressor(
    features_col="features",
    label_col="arr_delay",
    prediction_col="prediction",
    objective="reg:squarederror",
    eval_metric="rmse",
    num_workers=2,
)

pipeline = Pipeline(stages=[assembler, xgb])

# 6. Hyperparameter Grid & Fitting
param_grid = (
    ParamGridBuilder()
    .addGrid(xgb.max_depth, [4, 6])
    .addGrid(xgb.subsample, [0.8, 1.0])
    .addGrid(xgb.n_estimators, [300])
    .build()
)

evaluator = RegressionEvaluator(
    labelCol="arr_delay", predictionCol="prediction", metricName="rmse"
)

# Train the model
pipeline_model = pipeline.fit(train_df)

# 7. Evaluate on Test Data
predictions = pipeline_model.transform(test_df)
test_rmse = evaluator.evaluate(predictions)
test_r2 = evaluator.setMetricName("r2").evaluate(predictions)
test_mae = evaluator.setMetricName("mae").evaluate(predictions)

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")
print(f"Test R²:   {test_r2:.4f}")

INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 2 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'eval_metric': 'rmse', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Test RMSE: 22.8242
Test MAE:  10.1751
Test R²:   0.8680


In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from xgboost.spark import SparkXGBRegressor

# 1. Define Features (Matching your exact schema)
feature_cols = [
    "crs_dep_time",
    "crs_arr_time",
    "crs_elapsed_time",
    "distance",
    "dep_delay",
    "is_weekend",
    "temperature_2m",
    "precipitation",
    "snowfall",
    "weather_code",
    "wind_speed_10m",
    "wind_gusts_10m",
    "surface_pressure",
    "pressure_msl",
    "op_unique_carrier_index",
    "origin_index",
    "dest_index",
]

# 2. Vector Assembler to combine feature columns into a vector
assembler = VectorAssembler(
    inputCols=feature_cols, outputCol="features", handleInvalid="skip"
)

# 3. Native Spark XGBoost Regressor
xgb = SparkXGBRegressor(
    features_col="features",
    label_col="arr_delay",
    prediction_col="prediction",
    objective="reg:squarederror",
    eval_metric="rmse",
    n_estimators=300,
    random_state=42,
    num_workers=2,
)

# 4. Create Pipeline
pipeline = Pipeline(stages=[assembler, xgb])

# 5. PySpark Train/Test Split (80% Train, 20% Test)
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# 6. Define Param Grid (Equivalent to GridSearchCV param_grid)
param_grid = (
    ParamGridBuilder()
    .addGrid(xgb.max_depth, [4, 6])
    .addGrid(xgb.subsample, [0.8, 1.0])
    .build()
)

# 7. Evaluators
evaluator_rmse = RegressionEvaluator(
    labelCol="arr_delay", predictionCol="prediction", metricName="rmse"
)
evaluator_mae = RegressionEvaluator(
    labelCol="arr_delay", predictionCol="prediction", metricName="mae"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="arr_delay", predictionCol="prediction", metricName="r2"
)

# 8. Cross-Validation (3-Fold CV equivalent to scikit-learn cv=3)
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    parallelism=2,
    seed=42,
)

print("Starting Distributed XGBoost Cross-Validation training...")
cv_model = cv.fit(train_df)

# 9. Evaluate Best Model on Test Set
best_pipeline = cv_model.bestModel
predictions = best_pipeline.transform(test_df)

test_rmse = evaluator_rmse.evaluate(predictions)
test_mae = evaluator_mae.evaluate(predictions)
test_r2 = evaluator_r2.evaluate(predictions)

print("\n--- Model Evaluation Results ---")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")
print(f"Test R²:   {test_r2:.4f}")

# 10. Display Sample Predictions
predictions.select("arr_delay", "prediction", "dep_delay").show(10)

Starting Distributed XGBoost Cross-Validation training...


INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 2 workers with
	booster params: {'device': 'cpu', 'eval_metric': 'rmse', 'max_depth': 4, 'objective': 'reg:squarederror', 'random_state': 42, 'subsample': 1.0, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 2 workers with
	booster params: {'device': 'cpu', 'eval_metric': 'rmse', 'max_depth': 4, 'objective': 'reg:squarederror', 'random_state': 42, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 2 workers with
	booster params: {'device': 'cpu', 'eval_metric': 'rmse', 'max_depth': 6, 'objective': 'reg:squarederror', 'random_state': 42, 'subsample': 0.8, 'nthread': 1}



--- Model Evaluation Results ---
Test RMSE: 22.8658
Test MAE:  10.2029
Test R²:   0.8675
+---------+-------------------+---------+
|arr_delay|         prediction|dep_delay|
+---------+-------------------+---------+
|    -12.0| -11.45397663116455|     -7.0|
|      9.0|  3.520930290222168|     10.0|
|    -44.0|-17.774494171142578|    -11.0|
|    -24.0| -21.64743423461914|    -11.0|
|      9.0|   14.3997220993042|     21.0|
|    -21.0|-13.106863021850586|    -10.0|
|      1.0|-13.727675437927246|     -5.0|
|     31.0| 28.963167190551758|     32.0|
|     -2.0|-11.814192771911621|     -1.0|
|    -32.0| -19.83823585510254|     -7.0|
+---------+-------------------+---------+
only showing top 10 rows



In [10]:
model_path = "hdfs://namenode:9000/flight_project/gold/xgboost_flight_delay_model"

print(f"Saving the best XGBoost Pipeline model to HDFS: {model_path}")
# Save the 'best_pipeline' specifically, not the whole CV object
best_pipeline.write().overwrite().save(model_path)
print("✅ Best model successfully saved to HDFS!")

Saving the best XGBoost Pipeline model to HDFS: hdfs://namenode:9000/flight_project/gold/xgboost_flight_delay_model
✅ Best model successfully saved to HDFS!
